In [23]:
import requests
import pandas as pd
import joblib
import numpy as np

# === CONFIG ===
BASE_URL = "https://sdp-prem-prod.premier-league-prod.pulselive.com/api/v2/matches"
competition_id = 8
season_id = 2025

output_all = "../data/raw/football_data/premier_league_2025_26_fixtures.csv"
output_upcoming = "../data/processed/premier_league_2025_26_upcoming_prediction_template.csv"
output_annotated = "../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv"

feature_path = "../models/weekly/homewin/feature_list_20260327T205139Z.pkl"

all_rows = []
for mw in range(1, 39):
    params = {
        "competition": competition_id,
        "season": season_id,
        "matchweek": mw,
        "_limit": 20
    }
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(BASE_URL, params=params, headers=headers)
    js = resp.json()
    for match in js.get("data", []):
        home = match.get("homeTeam", {}).get("name", "")
        away = match.get("awayTeam", {}).get("name", "")
        home_score = match.get("homeTeam", {}).get("score", "")
        away_score = match.get("awayTeam", {}).get("score", "")
        kickoff = match.get("kickoff", "")
        venue = match.get("ground", "")
        matchweek = match.get("matchWeek", "")
        match_id = match.get("matchId", "")
        all_rows.append({
            "MatchWeek": matchweek,
            "HomeTeam": home,
            "AwayTeam": away,
            "HomeScore": home_score,
            "AwayScore": away_score,
            "Kickoff": kickoff,
            "Venue": venue,
            "MatchId": match_id
        })

df_all = pd.DataFrame(all_rows)
df_all.to_csv(output_all, index=False)

mask_unplayed = (
    (df_all["HomeScore"].isna() | (df_all["HomeScore"].astype(str).str.strip() == "")) &
    (df_all["AwayScore"].isna() | (df_all["AwayScore"].astype(str).str.strip() == ""))
)
df_upcoming = df_all[mask_unplayed].copy()

if "Kickoff" in df_upcoming.columns:
    df_upcoming["Date"] = pd.to_datetime(df_upcoming["Kickoff"], errors="coerce").dt.date

feature_list = joblib.load(feature_path)

for col in feature_list:
    if col not in df_upcoming.columns:
        df_upcoming[col] = np.nan

df_model_ready = df_upcoming[feature_list]


extra_cols = ["HomeTeam", "AwayTeam", "Date"]
cols_to_include = [col for col in extra_cols if col in df_upcoming.columns] + list(df_model_ready.columns)

seen = set()
cols_final = []
for c in cols_to_include:
    if c not in seen:
        cols_final.append(c)
        seen.add(c)

df_annotated = df_upcoming[cols_final]

df_model_ready.to_csv(output_upcoming, index=False)
df_annotated.to_csv(output_annotated, index=False)

print(f"Model-ready prediction template saved to {output_upcoming}, shape: {df_model_ready.shape}")
print(f"Annotated prediction template (with teams/dates) saved to {output_annotated}, shape: {df_annotated.shape}")

print("\nSample annotated version:")
print(df_annotated.head().T)

Model-ready prediction template saved to ../data/processed/premier_league_2025_26_upcoming_prediction_template.csv, shape: (70, 183)
Annotated prediction template (with teams/dates) saved to ../data/processed/premier_league_2025_26_upcoming_with_teams_and_date.csv, shape: (70, 186)

Sample annotated version:
                            310         311                       312  \
HomeTeam                Arsenal   Brentford                   Burnley   
AwayTeam            Bournemouth     Everton  Brighton and Hove Albion   
Date                 2026-04-11  2026-04-11                2026-04-11   
B365H                       NaN         NaN                       NaN   
B365D                       NaN         NaN                       NaN   
...                         ...         ...                       ...   
AwayTeam_Watford            NaN         NaN                       NaN   
AwayTeam_West_Brom          NaN         NaN                       NaN   
AwayTeam_West_Ham           NaN  

/var/folders/rj/ybzbgv510dd6rzmm3p9vf0000000gn/T/ipykernel_88004/2738478108.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_upcoming[col] = np.nan
